In [22]:
import pandas as pd
import numpy as np

In [23]:
df = pd.read_excel("../muestra_terra/Terra_example.xlsx")

In [24]:
import re
from collections import Counter

# Asegúrate de tener cargado tu DataFrame como `df`
requests = df['Request'].dropna().astype(str).tolist()

# Patrones para detectar secciones dentro del texto
section_patterns = [
    r"in the ([A-Z\s]+?) section",
    r"in the ([A-Z\s]+?) page",
    r"on the ([A-Z\s]+?) section",
    r"on the ([A-Z\s]+?) page",
    r"in ([A-Z\s]+?) section",
    r"on ([A-Z\s]+?) page"
]

# Extraer y normalizar
found_sections = []
for text in requests:
    for pattern in section_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        found_sections.extend([m.strip().title() for m in matches])

# Contar ocurrencias
section_counts = Counter(found_sections)
print(section_counts.most_common(10))

[('Who We Are', 5), ('The Who We Are', 4), ('The', 1), ('Keeping The Header Copy The Same On Both', 1)]


In [ ]:
import pandas as pd
import random
from faker import Faker
from transformers import pipeline, set_seed
from datetime import date, datetime, timedelta

# Inicialización
fake = Faker()
set_seed(42)
generator = pipeline("text-generation", model="gpt2")

# Valores reales de Request Type y demás categorías
request_types = ['Copy Revision', 'Design Issues', 'Requested Change', 'New Item']
priorities = ['Low', 'Normal', 'High', 'Urgent']
sections = ['hero section', 'footer', 'about us page', 'contact form', 'pricing table', 'FAQ block']
status_choices = ["Complete", "In Progress", "In Queue"]
status_weights = [0.2, 0.5, 0.3]  # pesos para fechas de 2025 o posteriores

priority_weights_by_type = {
    'Copy Revision':      ['Low'] * 6 + ['Normal'] * 3 + ['High'] * 1,
    'Design Issues':      ['Normal'] * 4 + ['High'] * 3 + ['Urgent'] * 2 + ['Low'] * 1,
    'Requested Change':   ['Normal'] * 3 + ['High'] * 4 + ['Urgent'] * 2 + ['Low'] * 1,
    'New Item':           ['High'] * 4 + ['Urgent'] * 4 + ['Normal'] * 2,
}

# Función para generar texto con GPT-2
def generate_request(request_type, section, priority):
    urgency_context = {
        'Low': "",
        'Normal': " This should be addressed soon.",
        'High': " This is important and needs to be prioritized.",
        'Urgent': " This is urgent and blocking progress. Please resolve immediately."
    }
    prompt_dict = {
        'Copy Revision': f"Please revise the copy in the {section}.",
        'Design Issues': f"There’s a layout or design issue in the {section}.",
        'Requested Change': f"The client has requested a change in the {section}.",
        'New Item': f"A new feature needs to be added to the {section}."
    }
    base_prompt = prompt_dict.get(request_type, f"Please update the {section}.")
    urgency = urgency_context.get(priority, "")
    full_prompt = base_prompt + urgency + " The request is:"
    output = generator(full_prompt, max_length=60, num_return_sequences=1, do_sample=True, temperature=0.9, top_p=0.95)[0]['generated_text']
    return output[len(full_prompt):].strip()

# Tema clientes y tamaños
num_clients = 25
client_names = list({fake.company() for _ in range(num_clients * 2)})[:num_clients]
random.shuffle(client_names)
clients = {}
for i, name in enumerate(client_names):
    if i < 5:
        clients[name] = 'large'
    elif i < 15:
        clients[name] = 'medium'
    else:
        clients[name] = 'small'
size_weights = {'large': 5, 'medium': 3, 'small': 1}
weighted_clients = []
for client, size in clients.items():
    weighted_clients.extend([client] * size_weights[size])

# Generación de proyectos por cliente con lógica según tamaño
project_suffixes = ['Platform', 'Revamp', 'Redesign', 'Campaign', 'Website', 'Landing', 'Portal', 'Dashboard']
projects_per_client = {}
for client, size in clients.items():
    num_projects = {
        'large': random.randint(5, 8),
        'medium': random.randint(3, 5),
        'small': random.randint(1, 2)
    }[size]
    projects = set()
    while len(projects) < num_projects:
        base = fake.word().capitalize()
        suffix = random.choice(project_suffixes)
        color = fake.color_name().replace(" ", "")
        project_name = f"{base} {suffix} – {color}"
        projects.add(project_name)
    projects_per_client[client] = list(projects)

# Fechas base por proyecto
start_date = date(2020, 1, 1)
end_date = date(2025, 1, 1)
project_date_bases = {}
for project_list in projects_per_client.values():
    for project in project_list:
        project_date_bases[project] = fake.date_between(start_date=start_date, end_date=end_date)

def generate_nearby_date(base_date, max_delta_days=45):
    delta = random.randint(-max_delta_days, max_delta_days)
    return (base_date + timedelta(days=delta)).strftime("%d/%m/%Y")

# Browser
weighted_browsers = (
    ["Chrome"] * 50 + ["Safari"] * 20 + ["Firefox"] * 15 + ["Edge"] * 8 +
    ["Brave"] * 3 + ["Opera"] * 2 + ["Internet Explorer"] * 1 + ["Samsung Internet"] * 1
)

# Devices
devices_weighted = ["Desktop"] * 50 + ["Mobile"] * 45 + ["Tablet"] * 5

# Generador de página web
def generate_client_page(company_name):
    domain = company_name.lower().replace(" ", "").replace(",", "").replace(".", "")
    return f"https://www.{domain}.com"

#Cosas de tiempo
estimated_time_options = [1, 2, 3, 5, 8, 13]
estimated_time_weights = {
    'Copy Revision':      [0.5, 0.3, 0.1, 0.05, 0.03, 0.02],
    'Design Issues':      [0.1, 0.2, 0.3, 0.25, 0.1, 0.05],
    'Requested Change':   [0.05, 0.1, 0.3, 0.3, 0.15, 0.1],
    'New Item':           [0.02, 0.05, 0.1, 0.3, 0.3, 0.23],
}

# Crear datos sintéticos
data = []
for _ in range(10000):
    request_type = random.choice(request_types)
    section = random.choice(sections)
    client = random.choice(weighted_clients)
    project = random.choice(projects_per_client[client])
    base_date = project_date_bases[project]
    input_date = generate_nearby_date(base_date)
    priority = random.choice(priority_weights_by_type[request_type])
    page = generate_client_page(client)

    weights = estimated_time_weights[request_type]
    estimated_time = random.choices(estimated_time_options, weights=weights, k=1)[0]
    real_time = max(1, round(random.normalvariate(estimated_time, 0.3)))
    input_dt = datetime.strptime(input_date, "%d/%m/%Y")

    # Status
    if input_dt.year < 2025:
        status_value = "Complete"
    else:
        status_value = random.choices(status_choices, weights=status_weights, k=1)[0]


        # Calcular real_time según estado
    if status_value == "Complete":
        real_time = max(1, round(random.normalvariate(estimated_time, estimated_time * 0.3)))
    elif status_value == "In Progress":
        real_time = round(estimated_time * random.uniform(1.0, 1.5))
    else:  # In Queue
        real_time = 0


    data.append({
        "Company": client,
        "Project Name": project,
        "Input Date": input_date,
        "Status": status_value,
        "Requester": fake.name(),
        "Request Type": request_type,
        "Priority": priority,
        "Request": generate_request(request_type, section, priority),
        "Device": random.choice(devices_weighted),
        "Browser": random.choice(weighted_browsers),
        "Page": page,
        "Estimated Time (tokens)": estimated_time,
        "Real Time": real_time,
    })

# Convertir a DataFrame
df = pd.DataFrame(data)

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eo

In [62]:
df

,Company,Project Name,Input Date,Status,Requester,Request Type,Priority,Request,Device,Browser,Page,Estimated Time (tokens),Real Time
0,Lopez-Espinoza,Push Revamp – Yellow,07/02/2023,Complete,David Williams,New Item,High,"1) Allow a footer with multiple pages, or 3-4 ...",Mobile,Firefox,https://www.lopez-espinoza.com,3,3
1,Brown Ltd,At Landing – Green,06/04/2021,Complete,Mary Hall,Copy Revision,Low,https://dnd.xbmc.org/content/dam/N2HVw8e9.pdf ...,Desktop,Chrome,https://www.brownltd.com,1,1
2,"Cummings, Daniel and Gonzales",Necessary Campaign – Silver,29/02/2020,Complete,David Peterson,New Item,Normal,We hope to have this bug fixed on the launch o...,Mobile,Chrome,https://www.cummingsdanielandgonzales.com,13,13
3,"Evans, Buchanan and Robinson",Total Redesign – Gold,13/02/2021,Complete,David Gates,Design Issues,Urgent,https://www.facebook.com/penguinboard/videos/7...,Mobile,Chrome,https://www.evansbuchananandrobinson.com,2,3
4,Miller-Peterson,Apply Platform – Gray,15/10/2024,Complete,Laura Mack,Design Issues,Urgent,’️We apologize for the inconvenience this has ...,Mobile,Chrome,https://www.miller-peterson.com,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,"Cummings, Daniel and Gonzales",See Website – BlueViolet,29/07/2021,Complete,Daniel Dunn,Design Issues,Urgent,https://bitbucket.org/shirok/sabu4/issues/26\n...,Mobile,Chrome,https://www.cummingsdanielandgonzales.com,2,2
996,Howard and Sons,National Campaign – DarkCyan,01/10/2024,Complete,Alexa Chapman,Copy Revision,Low,"If you would like to re-submit a work, e.g. if...",Mobile,Chrome,https://www.howardandsons.com,1,2
997,Howard and Sons,Recognize Redesign – Cyan,23/11/2024,Complete,Alexander Howell,Copy Revision,Low,Please use the following language to specify w...,Desktop,Safari,https://www.howardandsons.com,1,1
998,Frazier LLC,Property Landing – Red,28/10/2023,Complete,Neil Johnson,Requested Change,High,* a link to a website that allows you to set u...,Mobile,Safari,https://www.frazierllc.com,3,3


In [63]:
# Número de proyectos por empresa
proyectos_por_empresa = df[['Company', 'Project Name']].drop_duplicates()
proyectos_count = proyectos_por_empresa.groupby('Company').size().reset_index(name='Num Projects')

# Número de requests por proyecto
requests_count = df.groupby(['Company', 'Project Name']).size().reset_index(name='Num Requests')

# Merge para unir ambos
combined = requests_count.merge(proyectos_count, on='Company')

# Mostrar ordenado por empresa y luego por cantidad de requests
combined.sort_values(['Num Projects', 'Num Requests'], ascending=[False, False])

,Company,Project Name,Num Requests,Num Projects
23,"Evans, Buchanan and Robinson",Though Revamp – MediumOrchid,14,7
24,"Evans, Buchanan and Robinson",Total Redesign – Gold,12,7
19,"Evans, Buchanan and Robinson",Gun Campaign – DarkOrchid,11,7
21,"Evans, Buchanan and Robinson",Most Landing – DarkSalmon,11,7
22,"Evans, Buchanan and Robinson",Population Website – PaleGoldenRod,11,7
...,...,...,...,...
49,Lopez-Espinoza,Push Revamp – Yellow,22,1
65,Potts-Bates,Standard Website – Orchid,18,1
34,Hayes-Sutton,Discussion Revamp – MediumTurquoise,17,1
68,Ramos-Pena,Call Redesign – DarkCyan,16,1


In [64]:
#df2 = df[["Request Type" , "Status", "Input Date", "Requester", "Device", "Browser", "Request", "Page"]]

df.to_csv("synthetic_example.csv", index=False)

df.to_json("synthetic_example.json", orient="records", lines=True)